#1. Uploading the Dataset

We began by uploading the training and test datasets provided for the Kaggle housing price prediction competition. The training dataset contains the house features and their corresponding sale prices, while the test dataset contains the house features for which we need to predict sale prices.

We used the training dataset to develop and evaluate our regression models and the test dataset to generate predictions for our final Kaggle submission.

In [ ]:
# Upload the competition files into this Colab session.
from google.colab import files
uploaded = files.upload()

Saving train.csv to train (1).csv
Saving test.csv to test (1).csv


# 2. Importing Libraries and Loading the Data

We imported the Python libraries and tools required for data manipulation, preprocessing, model development, and evaluation.

We used pandas and NumPy to work with the data, while scikit-learn provides the preprocessing tools, regression estimators, and evaluation metrics.

Next, we loaded the training and test datasets and inspected their dimensions and first few rows to understand their structure.

In [ ]:
# Load the libraries used for data preparation, modeling and evaluation.
import numpy as np
import pandas as pd

from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Read known sale prices from training data and features from test data.
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Check the dataset sizes and inspect a few training examples.
print("Training data:", train.shape)
print("Test data:", test.shape)
print(train.head())

Training data: (1200, 81)
Test data: (260, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  S

We also examined the training dataset to identify columns containing missing values and understand how much data was missing in each column.

In [ ]:
# List training columns that contain missing values.
train.columns[train.isna().sum() > 0]

Index(['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish',
       'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'],
      dtype='object')

In [ ]:
# Measure how much of each incomplete column is missing.
train[train.columns[train.isna().sum() > 0]].isna().sum() / len(train)

,0
LotFrontage,0.175000
Alley,0.937500
MasVnrType,0.597500
MasVnrArea,0.005000
BsmtQual,0.026667
BsmtCond,0.026667
BsmtExposure,0.027500
BsmtFinType1,0.026667
BsmtFinType2,0.027500
FireplaceQu,0.470000


# 3. Separating the Target Variable and Features

Before training our models, we separated the target variable from the predictor features.

Our target variable is `SalePrice`, since the objective of the competition is to predict the selling price of each house.

We removed `Id` from the predictor features because it serves as an identifier rather than a meaningful property characteristic.

We also converted `MSSubClass` into a categorical variable because its numerical values represent dwelling types rather than quantities.

Finally, we ensured that the test dataset contains the same features in the same order as the training dataset.

In [ ]:
# Separate the price to predict from the house features.
target_column = "SalePrice"
id_column = "Id"

X = train.drop(columns=[target_column, id_column]).copy()
y = train[target_column].copy()

# Keep the test features and set aside their identifier.
X_test = test.drop(columns=[id_column]).copy()

# MSSubClass is a numeric code for a dwelling type, so treat it as a category.
X["MSSubClass"] = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

# Keep the test features in the same column order as the training features.
X_test = X_test[X.columns]

# Review the feature count and range of sale prices.
print("Features:", X.shape[1])
print("SalePrice summary:")
print(y.describe())

Features: 79
SalePrice summary:
count      1200.000000
mean     181414.628333
std       81070.908544
min       34900.000000
25%      129900.000000
50%      163700.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


# 4. Splitting the Training Data

We split the training dataset into training and validation sets to evaluate how well our models perform on unseen data.

We used 80% of the available training observations to train the models and reserved 20% for validation.

We set `random_state=42` to ensure that the split is reproducible, allowing us to compare different models using the same training and validation observations.

We also identified the numerical and categorical features separately because they require different preprocessing techniques.

In [ ]:
# Hold out some labeled houses to compare models on unseen rows.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Separate numbers and categories for suitable preprocessing.
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))

Numeric features: 35
Categorical features: 44


# 5. Data Preprocessing

Before fitting the regression models, we needed to handle missing values and convert categorical variables into a numerical format.

We used `SimpleImputer` to handle missing values. For numerical features, we used the strategy of replacing missing values with the column mean. For categorical features, we replaced missing values with the label "Missing".

We then used `OneHotEncoder` to convert categorical variables into numerical indicator columns.

We organized these preprocessing steps into a pipeline so that the same transformations are applied consistently during model training and prediction. We also used `handle_unknown="ignore"` to handle categories that may appear in the validation or test data but were not present in the training data.

In [ ]:
def make_preprocessor():
    # Fill missing category values and encode them for modeling.
    categorical_steps = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Apply numeric and category preparation to their matching columns.
    return make_column_transformer(
        (SimpleImputer(), numeric_columns),
        (categorical_steps, categorical_columns)
    )

# 6. Comparing a baseline and three models

We first made a simple baseline. It predicts the average SalePrice from the training portion for every house in the validation portion. This gave us a basic result to compare against.

We compared three models:

1. Linear Regression
2. Random Forest
3. Gradient Boosting

We included Random Forest because it was covered in class. It averages predictions from many decision trees, so it can pick up patterns in the housing features that may not follow a straight line. We used 300 trees and set the minimum leaf size to two houses, so the prediction is averaged across many trees and each leaf has more than one training example.

The competition instructions allowed us to use any scikit learn estimator, so we also tried Gradient Boosting. It builds trees one at a time, with each new tree trying to improve on errors made by earlier trees. We wanted to compare this approach with Random Forest and Linear Regression.

We used the same preprocessing steps for each model to fill in missing values and convert categories into numbers. We compared their results using Root Mean Squared Error, or RMSE. A lower RMSE means the predictions were closer to the actual SalePrice values overall. Larger errors affect RMSE more than smaller errors.

In [ ]:
# Compare a simple benchmark with the three regression models.
# Baseline: predict the average SalePrice from the training portion.
baseline_predictions = np.ones(len(y_valid)) * y_train.mean()
baseline_rmse = np.sqrt(mean_squared_error(y_valid, baseline_predictions))

# Store validation RMSE values so the approaches can be compared.
scores = {"Mean baseline": baseline_rmse}

# Pair each estimator with the same preprocessing steps.
models = {
    "Linear Regression": Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", LinearRegression())
    ]),

    "Gradient Boosting": Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", GradientBoostingRegressor(random_state=42))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        # Average many trees and require at least two houses in each leaf.
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1  # Uses available CPU cores to speed up the forest.
        ))
    ])
}

# Fit each model on the training split and score held out rows.
for name, model in models.items():
    model.fit(X_train, y_train)
    validation_predictions = model.predict(X_valid)

    # RMSE is the square root of mean squared error, measured in dollars.
    rmse = np.sqrt(mean_squared_error(y_valid, validation_predictions))
    scores[name] = rmse

    print(f"{name} validation RMSE: ${rmse:,.2f}")

# Identify the result with the smallest validation error.
best_name = min(scores, key=scores.get)
print(f"\nBest validation result: {best_name} (${scores[best_name]:,.2f} RMSE)")

Linear Regression validation RMSE: $36,689.04
Gradient Boosting validation RMSE: $39,950.61
Random Forest validation RMSE: $37,182.80

Best validation result: Linear Regression ($36,689.04 RMSE)


# 7. Comparing validation and Kaggle results

On our validation split, Linear Regression had the lowest RMSE at 36,689.04 dollars. Random Forest had an RMSE of 37,182.80 dollars, and Gradient Boosting had an RMSE of 39,950.61 dollars.

Our Kaggle public scores ranked the models differently. Linear Regression scored 61,665.19, Random Forest scored 36,255.59, and Gradient Boosting scored 24,318.95. We chose Gradient Boosting for our final Kaggle submission because it had the lowest public RMSE among our submissions.

In [ ]:
# Show which model did best on this validation split.
validation_best_name = min(models, key=lambda name: scores[name])

print("Best model on the validation split:", validation_best_name)
print("Validation RMSE:", f"{scores[validation_best_name]:,.2f}", "dollars")

Best model on the validation split: Linear Regression
Validation RMSE: 36,689.04 dollars




# 8. Final model selection and training

Our validation split favored Linear Regression, but our Kaggle submissions favored Gradient Boosting. We used Gradient Boosting for the final submission because it had the lowest Kaggle public RMSE. We trained it on all the training data before predicting prices for the test rows.





In [ ]:
# Refit the selected model using all houses with known sale prices.
final_model_name = "Gradient Boosting"
final_model = models[final_model_name]

final_model.fit(X, y)
# Predict prices in the same row order as the test features.
test_predictions = final_model.predict(X_test)

print("Final model:", final_model_name)
print("Number of test predictions:", len(test_predictions))

Final model: Gradient Boosting
Number of test predictions: 260


# 9. Exporting the Final Predictions

The submission file contains 260 predictions. Kaggle required submission IDs from 0 through 259, so we used those IDs and kept the predictions in the same order as the test rows.

In [ ]:
# Create the two columns Kaggle expects for the submission.
submission = pd.DataFrame({
    "Id": np.arange(len(test_predictions)),
    "SalePrice": test_predictions
})

# Save without adding pandas' own row number as an extra column.
submission.to_csv("submission_gradient_boosting.csv", index=False)

# Check the row count, sample predictions and missing values.
print(submission.shape)  # should be (260, 2)
print(submission.tail())  # IDs should end at 259
print(submission.isna().sum())

# Download the completed submission file from Colab.
files.download("submission_gradient_boosting.csv")

(260, 2)
      Id      SalePrice
255  255  168928.741192
256  256  212689.630330
257  257  257276.124025
258  258  140696.593110
259  259  146168.673034
Id           0
SalePrice    0
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>